# TP — CNN & Transfer Learning
### PyTorch · Jupyter / Colab

**Nom :** ..............................................................  
**Prénom :** ..........................................................  
**Date :** ..............................................................

## Objectifs

Construire un ConvNet simple sur MNIST, mesurer l’impact des augmentations, puis réaliser un **transfer learning** avec `timm` et un ResNet-18.

Le notebook est volontairement **à trous** : les cellules marquées `TODO` sont à compléter.  
Les questions d’analyse sont laissées sans réponse.

### Ressources utiles
- `torch.nn.Conv2d` — documentation PyTorch
- `torchvision.transforms` — documentation torchvision
- `timm` — documentation et modèles pré-entraînés

## Rappel express — CNN

Une couche `Conv2d` applique \(C_{\mathrm{out}}\) filtres de taille \(k\times k\) à une entrée comportant \(C_{\mathrm{in}}\) canaux.

Pour une dimension spatiale d’entrée \(H\), la taille de sortie vaut :

\[
H_{\mathrm{out}}
=
\left\lfloor
\frac{H + 2P - k}{s}
\right\rfloor + 1,
\]

où \(P\) est le *padding* et \(s\) le *stride*.

Les opérations usuelles dans ce TP sont : convolution, ReLU, MaxPool, puis une tête MLP.

# 0) Préparation — environnement et utilitaires

GPU recommandé. Si `timm` n’est pas installé, décommentez la première cellule.

In [ ]:
# !pip install --quiet timm==1.0.9

In [ ]:

import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import timm
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

### Métriques et évaluation

Les fonctions suivantes sont fournies.

In [ ]:

def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()


@torch.no_grad()
def evaluate(model, loader, loss_fn=nn.CrossEntropyLoss()):
    model.eval()
    tot_loss, tot_acc, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        b = y.size(0)
        tot_loss += loss.item() * b
        tot_acc += (logits.argmax(1) == y).float().sum().item()
        n += b

    return tot_loss / n, tot_acc / n

### À faire — boucle d’entraînement

Complétez les quelques trous manquants dans la boucle d’entraînement.  
Décrivez en commentaire à quoi correspondent les pièces manquantes.

In [ ]:

def train(model, train_loader, val_loader, epochs=5, lr=1e-3, wd=0.0):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    # TODO — choisir la fonction de coût adaptée
    loss_fn = ...

    best, best_state = math.inf, None

    for ep in range(1, epochs + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}")

        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = loss_fn(logits, y)

            opt.zero_grad()

            # TODO — étape manquante de rétropropagation
            ...

            opt.step()

            pbar.set_postfix(loss=f"{loss.item():.3f}")

        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        print(f"Val | loss: {val_loss:.4f} | acc: {val_acc * 100:.2f}%")

        if val_loss < best:
            best = val_loss
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(
            {k: v.to(device) for k, v in best_state.items()}
        )

### Commentaires sur les éléments complétés

Écrivez ici, en quelques lignes, le rôle des instructions que vous avez ajoutées.

**Réponse :**

...

# 1) ConvNet basique sur MNIST

MNIST contient 60 000 images d’entraînement et 10 000 images de test, en niveaux de gris, de taille \(28	imes28\).

Normalisez avec les statistiques indiquées dans l’énoncé :

\[
\mu = 0{,}1307,
\qquad
\sigma = 0{,}3081.
\]

Séparez également un jeu de validation.

### Chargement des données

In [ ]:

BATCH = 128  # Ajuster si besoin / mémoire insuffisante

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_full = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

test_set = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

VAL_RATIO = 0.10

n_val = int(len(train_full) * VAL_RATIO)
n_train = len(train_full) - n_val

train_set, val_set = random_split(
    train_full,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

# Astuce Windows : supprimer num_workers en cas de plantage du noyau.
train_loader = DataLoader(
    train_set,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

len(train_set), len(val_set), len(test_set)

## Modèle

Construisez un réseau avec deux blocs `Conv2d + ReLU + MaxPool`.

### À faire
Complétez :
- le nombre de canaux d’entrée ;
- les tailles de noyau ;
- le nombre de canaux de la seconde convolution ;
- le padding ;
- la dimension d’entrée du premier `Linear`.

Expliquez en commentaire comment vous avez obtenu la dimension de la carte finale.

In [ ]:

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=...,
                out_channels=32,
                kernel_size=...,
                padding=1,
            ),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 28 -> 14

            nn.Conv2d(
                32,
                ...,
                kernel_size=...,
                padding=...,
            ),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # TODO — déterminer la dimension spatiale finale
        )

        self.head = nn.Sequential(
            nn.Flatten(),

            # TODO — calculer la dimension d'entrée
            nn.Linear(64 * ... * ..., 128),

            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # TODO — compléter le forward pass
        ...
        ...
        return ...

### Vérification rapide de l’architecture

Vous pouvez utiliser cette cellule pour inspecter le modèle avant l’entraînement.

In [ ]:

model = SmallCNN().to(device)
model

### Entraînement et test

In [ ]:

model = SmallCNN().to(device)

train(
    model,
    train_loader,
    val_loader,
    epochs=5,
    lr=1e-3,
    wd=1e-4,
)

test_loss, test_acc = evaluate(model, test_loader)
print(f"Test | loss: {test_loss:.4f} | acc: {test_acc * 100:.2f}%")

### Question 1

Que se passe-t-il si vous remplacez `MaxPool2d(2)` par un `stride=2` dans la convolution précédente ?

**Réponse :**

...

### Question 2

Tracez l’évolution des *loss* en entraînement et en validation.

Adaptez si nécessaire la fonction `train` pour conserver l’historique des valeurs.

In [ ]:

# TODO — modifier si nécessaire la boucle d'entraînement
# afin de stocker les loss train/validation.

...

In [ ]:

# TODO — tracer les courbes train / validation.

...

### Question 3

Ajoutez un bloc convolution–ReLU de votre choix dans `self.features`.

Identifiez ce qu’il faut ensuite réajuster dans l’architecture.

**Observations :**

...

In [ ]:

# TODO — définir ici une variante du réseau avec un bloc supplémentaire.

...

### Bonus — CIFAR-10

Examinez le dataset CIFAR-10.  
Quelles parties de l’architecture ou du pipeline de données faut-il adapter pour que le réseau fonctionne ?

**Réponse :**

...

In [ ]:

# Espace de travail optionnel — CIFAR-10

...

# 2) Effet des augmentations — `torchvision`

Les augmentations servent à simuler des variations des données d’entrée : rotations, translations, bruit, effacement, etc.

### À faire

Choisissez **2 à 3 transformations pertinentes pour MNIST** et paramétrez-les.  
Justifiez vos choix en une ou deux phrases.

In [ ]:

aug_transform = transforms.Compose([
    # TODO — choisir et paramétrer 2 à 3 transformations pertinentes
    # TODO — réfléchir également à leur ordre
    ...,

    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

### Justification des augmentations choisies

**Réponse :**

...

In [ ]:

train_full_aug = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=aug_transform,
)

n_val = int(len(train_full_aug) * VAL_RATIO)
n_train = len(train_full_aug) - n_val

train_set_aug, val_set_aug = random_split(
    train_full_aug,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader_aug = DataLoader(
    train_set_aug,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader_aug = DataLoader(
    val_set_aug,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

model_aug = SmallCNN().to(device)

train(
    model_aug,
    train_loader_aug,
    val_loader_aug,
    epochs=5,
    lr=1e-3,
    wd=1e-4,
)

test_loss_aug, test_acc_aug = evaluate(model_aug, test_loader)
print(
    f"[Aug] Test | loss: {test_loss_aug:.4f} "
    f"| acc: {test_acc_aug * 100:.2f}%"
)

## Analyse

Comparez les performances sans et avec augmentations.

1. Que se passe-t-il lorsque vous augmentez l’amplitude des rotations ?
2. Quelle augmentation semble la plus nuisible ?
3. Quelle augmentation semble la plus utile ?

**Réponse :**

...

In [ ]:

# Espace de travail — expériences complémentaires sur les augmentations

...

# 3) Transfer learning — ResNet-18 (`timm`) → MNIST

On réutilise un backbone pré-entraîné sur ImageNet, initialement prévu pour des images RGB de taille typique \(224	imes224\), puis on adapte la tête de classification à 10 classes.

Deux phases sont proposées :

1. geler le backbone et entraîner la tête ;
2. dégeler le réseau et effectuer un *fine-tuning* global avec un faible taux d’apprentissage.

## Pipeline d’entrée

### À faire

Préparez le pipeline :
- redimensionner les images en \(224	imes224\) ;
- dupliquer le canal pour passer de 1 à 3 canaux ;
- normaliser avec les statistiques ImageNet indiquées.

Expliquez en une phrase pourquoi la duplication du canal est nécessaire.

In [ ]:

IMG_SIZE = 224

tf_train_tl = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # TODO — convertir l'image dans le format attendu
    # TODO — passer de 1 canal à 3 canaux
    ...,

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

# TODO — vous pouvez différencier le pipeline train du pipeline validation/test
tf_eval_tl = tf_train_tl

### Pourquoi dupliquer le canal ?

**Réponse :**

...

In [ ]:

train_full_tl = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=tf_train_tl,
)

test_set_tl = datasets.MNIST(
    root="./data",
    train=False,
    download=False,
    transform=tf_eval_tl,
)

n_val = int(len(train_full_tl) * VAL_RATIO)
n_train = len(train_full_tl) - n_val

train_set_tl, val_set_tl = random_split(
    train_full_tl,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

### À faire — DataLoaders

Complétez les paramètres manquants.

In [ ]:

# Astuce Windows : mettre num_workers=0 si nécessaire.

train_loader_tl = DataLoader(
    ...,
    batch_size=...,
    shuffle=...,
    num_workers=2,
    pin_memory=True,
)

val_loader_tl = DataLoader(
    val_set_tl,
    batch_size=BATCH,
    shuffle=...,
    num_workers=2,
    pin_memory=True,
)

test_loader_tl = DataLoader(
    ...,
    batch_size=BATCH,
    shuffle=...,
    num_workers=2,
    pin_memory=True,
)

## Modèle et stratégie d’entraînement

Créez un `resnet18` pré-entraîné et remplacez sa tête par une tête de classification à 10 classes.

### Phase 1
Geler tous les poids sauf ceux de la tête.

### Phase 2
Dégeler ensuite le modèle pour un *fine-tuning* global léger.

Testez par exemple 3 + 2 époques, comme indiqué dans l’énoncé.

In [ ]:

# TODO — choisir correctement l'option pretrained
model_tl = timm.create_model(
    "resnet18",
    pretrained=...,
    num_classes=10,
)

model_tl

### Phase 1 — gel du backbone

Vérifiez le nom exact de la tête dans le modèle `timm`.

In [ ]:

# TODO — geler le backbone et laisser la tête entraînable

for name, p in model_tl.named_parameters():
    # TODO — identifier correctement les paramètres de la tête
    ...

In [ ]:

model_tl = model_tl.to(device)

# TODO — compléter les hyperparamètres
train(
    model_tl,
    train_loader_tl,
    val_loader_tl,
    epochs=...,
    lr=...,
    wd=...,
)

### Phase 2 — fine-tuning global

In [ ]:

for p in model_tl.parameters():
    # TODO — dégeler les couches
    p.requires_grad = ...

In [ ]:

train(
    model_tl,
    train_loader_tl,
    val_loader_tl,
    epochs=...,
    lr=5e-4,
    wd=1e-4,
)

### Question

Pourquoi utilise-t-on un taux d’apprentissage relativement faible pendant cette phase de *fine-tuning* ?

**Réponse :**

...

In [ ]:

test_loss_tl, test_acc_tl = evaluate(model_tl, test_loader_tl)

print(
    f"[ResNet18 TL] Test | loss: {test_loss_tl:.4f} "
    f"| acc: {test_acc_tl * 100:.2f}%"
)

## Discussion

Le transfer learning peut accélérer la convergence et introduire une forme de régularisation implicite.

Discutez les points suivants :

1. Quels risques apparaissent si l’écart de domaine entre ImageNet et le jeu cible devient trop important ?
2. Que changeriez-vous pour des images beaucoup plus variées ou éloignées du domaine source, par exemple CIFAR-10 ?

**Réponse :**

...

# Extensions optionnelles

Vous pouvez explorer les pistes suivantes :

- ajouter un `Dropout` après la couche cachée de la tête et mesurer son effet ;
- tester un scheduler, par exemple `CosineAnnealingLR` ;
- tester le *label smoothing* de `CrossEntropyLoss` ;
- comparer d’autres architectures `timm`, par exemple `efficientnet_b0` et `convnext_tiny` ;
- comparer le nombre de paramètres.

In [ ]:

# Extension 1 — Dropout

...

In [ ]:

# Extension 2 — scheduler / label smoothing

...

In [ ]:

# Extension 3 — autres architectures timm et nombre de paramètres

...

# Tableau de synthèse des expériences

Complétez ce tableau au fur et à mesure du TP.

| Expérience | Hyperparamètres principaux | Accuracy val. | Accuracy test | Observations |
|---|---|---:|---:|---|
| SmallCNN — baseline |  |  |  |  |
| SmallCNN — augmentations |  |  |  |  |
| ResNet-18 — tête seule |  |  |  |  |
| ResNet-18 — fine-tuning |  |  |  |  |
| Extension |  |  |  |  |

## Conseils pratiques

Sur Colab, activez le GPU via :

**Exécution → Modifier le type d’exécution → Accélérateur matériel = GPU**

Conservez une trace des expériences : hyperparamètres, accuracy de validation/test et observations.